# ZnO–CO2 simulation workflow

Run the cells from `notebooks/` or the repository root.
Every method runs through the official workflow (`scripts.workflows.run_simulations`)
with `configs/simulations/zno_production.yaml`, so the notebook and
`./run_simulations.sh` produce the same isolated run in
`results/simulations/<RUN_ID>/`. Numerical settings live in the YAML, not here.

## Local execution

One-time setup:

```bash
cd notebooks
./setup_lucas_env.sh
```

Then run notebook headless:

```bash
cd notebooks
QML_ZNO_SIM_ENV=<env> ./run_zno_notebook.sh zno-co2-lucas.ipynb run1.executed.ipynb
```

Or open Jupyter in `notebooks/` and run all cells top-to-bottom.
Progress of a running method: `python -m scripts.workflows.run_status <RUN_ID>`.
Post-processing (CO2RR corrections, MD trajectory analysis, NEB tables) and figures are
in `zno-co2-analysis.ipynb`, as with the launchers, so both routes produce the same run.

In [ ]:
from pathlib import Path
import subprocess
import sys
import os
import csv
import math

CWD = Path.cwd()
NOTEBOOK_DIR = next(
    (path for path in [CWD, CWD / "notebooks", CWD.parent / "notebooks"] if (path / "zno-co2-lucas.ipynb").exists()),
    CWD,
)
sim_candidates = [
    CWD,
    CWD.parent,
    NOTEBOOK_DIR.parent,
]
LUCAS_DIR = next((path for path in sim_candidates if (path / "scripts" / "workflows" / "run_simulations.py").exists()), None)
if LUCAS_DIR is None:
    raise FileNotFoundError(
        "Could not resolve simulation scripts. Run from notebooks/ or the repository root."
    )

print(f"Current directory: {CWD}")
print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Simulator directory: {LUCAS_DIR}")

def run_cmd(args: list[str], env: dict | None = None) -> None:
    """Run a command from the repository root and stream its output into the cell."""
    print("$", " ".join(args), flush=True)
    with subprocess.Popen(args, cwd=LUCAS_DIR, env=env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
    if process.returncode:
        raise subprocess.CalledProcessError(process.returncode, args)

def safe_float(value) -> float:
    try:
        return float(value)
    except (TypeError, ValueError):
        return float("nan")

def read_rows(path: Path) -> list[dict]:
    if not path.exists():
        print(f"[missing] {path}")
        return []
    with path.open("r", newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))

## Run Configuration

Edit only this cell. Numerical settings (cutoff, k-points, steps, seeds, NEB pairs)
are read from `CONFIG`; change them there so every run records them.

In [ ]:
RUN_ID = "zno_notebook_01"   # results/simulations/<RUN_ID>/; reuse it to continue a run
CONFIG = LUCAS_DIR / "configs" / "simulations" / "zno_production.yaml"
MATERIAL = "ZnO"
SITES = ["top_metal", "top_oxygen", "bridge"]
MPI_PROCESSES = 6            # 2x1 cell, k 2x3x1, symmetry off: 6 k-points, one per rank
RETRY_FAILED = False         # True: rerun a method that failed or was interrupted (archives its outputs)

RUN_GEOMETRY = False   # Initial slab and CO2 structures
RUN_DFT      = False   # GPAW relaxations and adsorption energies
RUN_PATHWAY  = False   # CO2RR (COOH*, CO*) free-energy pathway
RUN_TDDFT    = False   # LrTDDFT of the isolated CO2 in its adsorbed geometry
RUN_MD       = False   # Langevin AIMD (equilibration + production, all configured seeds)
RUN_NEB      = False   # NEB between the configured site pairs
RUN_SUMMARY  = False   # Site summary from DFT + CO2RR (no screening score)

RUN_DIR = LUCAS_DIR / "results" / "simulations" / RUN_ID
# The run manifest records the conda environment and --resume checks it, so a run must be
# continued from the same environment it was started in (on the lab PC: gpaw-dft). The
# kernel's bin directory goes first in PATH so the workflow finds this env's mpiexec/gpaw.
KERNEL_ENV = os.environ.get("QML_ZNO_SIM_ENV") or (
    Path(sys.prefix).name if Path(sys.prefix).parent.name == "envs" else os.environ.get("CONDA_DEFAULT_ENV", "")
)
WORKFLOW_ENV = {
    **os.environ,
    # Name, or last component of a prefix path, as the launchers record it.
    "QML_ZNO_ACTIVE_ENVIRONMENT": Path(KERNEL_ENV).name if KERNEL_ENV else "",
    "PATH": str(Path(sys.executable).parent) + os.pathsep + os.environ.get("PATH", ""),
}
print(f"Run directory: {RUN_DIR}" + ("  (exists: methods continue with --resume)" if RUN_DIR.is_dir() else ""))
print(f"Conda environment recorded in the run: {Path(KERNEL_ENV).name if KERNEL_ENV else '(configured default)'}")

def run_workflow(method: str) -> None:
    """One method through the workflow; prerequisites are added and completed ones skipped."""
    resuming = (RUN_DIR / "run_manifest.json").exists()
    # A run continues from its own config.yaml snapshot: resume checks the file hash, so
    # it refuses the reference YAML once that file has been edited after the run started.
    config = RUN_DIR / "config.yaml" if resuming and (RUN_DIR / "config.yaml").is_file() else CONFIG
    cmd = [
        sys.executable, "-m", "scripts.workflows.run_simulations", method,
        "--config", str(config), "--run-id", RUN_ID,
        "--materials", MATERIAL, "--sites", ",".join(SITES),
        "--mpi-processes", str(MPI_PROCESSES),
    ]
    if resuming:
        cmd.append("--resume")
        if RETRY_FAILED:
            cmd.append("--retry-failed")
    run_cmd(cmd, env=WORKFLOW_ENV)

## 0) Environment Check

This confirms `numpy`, `ase`, and `gpaw` are available in the active environment.

In [ ]:
run_cmd([sys.executable, "-c", "import numpy, ase, gpaw; print('env-ok')"])


## 1) Geometry

Generate the ZnO(10-10) 2x1 slab and the CO2 starting structures of the run.

In [ ]:
if RUN_GEOMETRY:
    run_workflow("geometry")

In [ ]:
if RUN_GEOMETRY:
    geometry_dir = RUN_DIR / "data" / "geometries" / MATERIAL
    for path in sorted(geometry_dir.glob("*")) if geometry_dir.is_dir() else []:
        print(f"[OK] {path}")
    if not geometry_dir.is_dir():
        print(f"[missing] {geometry_dir}")

## 2) DFT

Run DFT relaxation for clean ZnO and ZnO+CO2 adsorption cases.
This can take significant compute time.

In [ ]:
if RUN_DFT:
    run_workflow("dft")

In [ ]:
if RUN_DFT:
    zno_dft = [row for row in read_rows(RUN_DIR / "dft_summary.csv")
               if row.get("material") == MATERIAL and row.get("site") in SITES]
    for row in zno_dft:
        eads = safe_float(row.get("adsorption_energy_ev"))
        note = "  (E_ads > 0: not bound at PBE level)" if eads > 0 else ""
        print(
            f"{row['site']:>10} | status={row.get('dft_status', '')} "
            f"| Eads={eads:+.4f} eV | state={row.get('final_state', '')} "
            f"| OCO={safe_float(row.get('oco_angle_deg')):.2f} deg{note}"
        )
    print("Site differences of a few 0.01 eV are within the numerical uncertainty (~0.05 eV).")

## 3) CO2RR pathway

CO2(g) -> COOH* -> CO* + H2O -> CO(g) with the computational hydrogen electrode,
then the post-processing corrections (variant c: quasi-harmonic floor + PBE gas-phase
corrections). Sites whose intermediates desorb or change identity get no U_L.

In [ ]:
if RUN_PATHWAY:
    run_workflow("co2rr")

In [ ]:
if RUN_PATHWAY:
    corrected = RUN_DIR / "co2rr_corrected_summary.csv"
    if corrected.exists():
        print("Variant c (quasi-harmonic floor + PBE gas-phase corrections):")
        rows = [row for row in read_rows(corrected)
                if row.get("material") == MATERIAL and row.get("site") in SITES and row.get("variant") == "c"]
        status_key, prefix = "status", "dG"
    else:
        print("As computed (no corrections yet: run the CO2RR section of zno-co2-analysis.ipynb):")
        rows = [row for row in read_rows(RUN_DIR / "co2rr_pathway_summary.csv")
                if row.get("material") == MATERIAL and row.get("site") in SITES]
        status_key = "pathway_validity"
        prefix = "dG" if rows and f"dG_CO2_to_COOH_ev" in rows[0] else "dE"
    for row in rows:
        u_l = safe_float(row.get("limiting_potential_v"))
        print(
            f"{row['site']:>10} | {row.get(status_key, ''):<18} "
            f"| {prefix}(CO2->COOH*)={safe_float(row.get(f'{prefix}_CO2_to_COOH_ev')):+.3f} "
            f"| {prefix}(COOH*->CO*)={safe_float(row.get(f'{prefix}_COOH_to_CO_ev')):+.3f} "
            f"| {prefix}(CO des.)={safe_float(row.get(f'{prefix}_CO_desorption_ev')):+.3f} eV "
            f"| U_L={'n/a' if not math.isfinite(u_l) else f'{u_l:+.3f} V'}"
        )

## 4) TDDFT

LrTDDFT of the CO2 molecule in its adsorbed geometry, isolated in a box without the slab.
It is a gas-phase reference, not the optical response of ZnO-CO2; transitions above
the Kohn-Sham ionization threshold (-eps_HOMO) are discretized continuum states.

In [ ]:
if RUN_TDDFT:
    run_workflow("tddft")

In [ ]:
if RUN_TDDFT:
    rows = [row for row in read_rows(RUN_DIR / "tddft_summary.csv")
            if row.get("material") == MATERIAL and row.get("site") in SITES]
    for row in rows:
        bright = row.get("co2_isolated_first_bright_excitation_ev", row.get("co2_isolated_onset_ev"))
        print(
            f"{row['site']:>10} | status={row.get('tddft_status', '')} "
            f"| first={safe_float(row.get('co2_isolated_first_excitation_ev')):.3f} eV "
            f"| first bright={safe_float(bright):.3f} eV "
            f"| peak={safe_float(row.get('co2_isolated_peak_ev')):.3f} eV "
            f"| -eps_HOMO={safe_float(row.get('co2_isolated_ks_ionization_threshold_ev')):.3f} eV "
            f"| TRK={safe_float(row.get('trk_fraction')):.0%}"
        )

## 5) Molecular Dynamics - MD

Langevin AIMD of ZnO + CO2 from the optimized DFT geometries, one trajectory per
configured seed (equilibration, then production; settings in `CONFIG`).
The trajectory analysis reports, per trajectory, how long the CO2 stays at its
starting site, whether it leaves the surface and the lateral displacement.

In [ ]:
if RUN_MD:
    run_workflow("md")

In [ ]:
if RUN_MD:
    for site in SITES:
        for row in read_rows(RUN_DIR / "md" / MATERIAL / site / "md_site_summary.csv"):
            print(
                f"{site:>10} seed {row.get('seed', ''):>5} | {row.get('md_status', ''):<4} "
                f"| <T>={safe_float(row.get('temperature_k_mean')):.1f} K "
                f"| <C-slab>={safe_float(row.get('c_min_slab_distance_ang_mean')):.2f} A "
                f"| <O-C-O>={safe_float(row.get('oco_angle_deg_mean')):.1f} deg"
            )
    print("Site occupancy and desorption: MD section of zno-co2-analysis.ipynb.")

## 6) Nudged Elastic Band — NEB

NEB between the site pairs configured in `CONFIG` (neighbouring sites only:
top_metal -> bridge and bridge -> top_oxygen; top_metal -> top_oxygen passes over
the bridge). Barriers below the resolution, or bands whose endpoints are not minima
along the band, are reported without a barrier.

In [ ]:
if RUN_NEB:
    run_workflow("neb")
    for row in read_rows(RUN_DIR / "neb" / MATERIAL / "neb_paths_summary.csv"):
        resolved = row.get("barrier_resolved") == "True"
        print(
            f"{row.get('initial_site', ''):>10} -> {row.get('final_site', ''):<10} | {row.get('status', ''):<17} "
            f"| Ea fwd={safe_float(row.get('forward_barrier_eV')):.3f} eV"
            + ("" if resolved else " (not resolved)")
            + (" | endpoint not a minimum along the band" if row.get("endpoint_not_minimum") == "True" else "")
        )

## Site summary (DFT + CO2RR)

Adsorption energy and limiting potential per site. There is no screening score:
the site ordering is not resolved when the differences are within the numerical
uncertainty (~0.05 eV in E_ads, ~0.1 V in U_L).

In [ ]:
if RUN_SUMMARY:
    dft = {row["site"]: row for row in read_rows(RUN_DIR / "dft_summary.csv") if row.get("material") == MATERIAL}
    corrected = RUN_DIR / "co2rr_corrected_summary.csv"
    if corrected.exists():
        co2rr = {row["site"]: row for row in read_rows(corrected)
                 if row.get("material") == MATERIAL and row.get("variant") == "c"}
    else:  # as computed; corrections come from zno-co2-analysis.ipynb
        co2rr = {row["site"]: {**row, "status": row.get("pathway_validity", "")}
                 for row in read_rows(RUN_DIR / "co2rr_pathway_summary.csv") if row.get("material") == MATERIAL}
    for site in SITES:
        eads = safe_float(dft.get(site, {}).get("adsorption_energy_ev"))
        u_l = safe_float(co2rr.get(site, {}).get("limiting_potential_v"))
        print(
            f"{site:>10} | Eads={eads:+.3f} eV | CO2RR {co2rr.get(site, {}).get('status', 'n/a'):<18} "
            f"| U_L={'n/a' if not math.isfinite(u_l) else f'{u_l:+.3f} V'}"
        )